# 📈 Equity Analysis — Demo Notebook

This notebook is a **thin presentation layer** over the `stockanalysis` package.
All logic lives in the package (`src/stockanalysis/`); here we just call it and
render the results interactively.

Run **all cells top to bottom**. Requires live network access (Yahoo Finance).

## 0 · Setup

Make the package importable (editable install is preferred — see README — but this also works in-place) and enable inline Plotly.

In [1]:
import sys, logging
sys.path.insert(0, "../src")          # run without installing; or: pip install -e ..

import plotly.io as pio
pio.renderers.default = "notebook"     # inline interactive charts
logging.basicConfig(level=logging.INFO, format="%(message)s")

from pathlib import Path
import pandas as pd
import stockanalysis as sa
print("stockanalysis", sa.__version__)

stockanalysis 0.1.0


## 1 · Run the pipeline

`run()` does ingest → screen → indicators → signals and returns a `Results` object.

In [2]:
results = sa.run(out_dir="../output")   # add export_target="excel" / save_charts=True → run() persists into the same folder
screened_df   = results.screened_df
signal_matrix = results.signal_matrix
tech          = results.tech
out = Path(results.run_dir)            # this run's timestamped folder — §4/§5/§7 all write here
out.mkdir(parents=True, exist_ok=True)
print(f"{len(signal_matrix)} signals · {len(tech)} tickers with indicators · output → {out}/")

Fetching data for 18 tickers (hits Yahoo Finance once per ticker)...
  AAPL      752 rows (2023-08-14 -> 2026-08-12)
  MSFT      752 rows (2023-08-14 -> 2026-08-12)
  NVDA      752 rows (2023-08-14 -> 2026-08-12)
  GOOGL     752 rows (2023-08-14 -> 2026-08-12)
  META      752 rows (2023-08-14 -> 2026-08-12)
  AMD       752 rows (2023-08-14 -> 2026-08-12)
  ASML      752 rows (2023-08-14 -> 2026-08-12)
  MU        752 rows (2023-08-14 -> 2026-08-12)
  SNOW      752 rows (2023-08-14 -> 2026-08-12)
  U         752 rows (2023-08-14 -> 2026-08-12)
  TSM       752 rows (2023-08-14 -> 2026-08-12)
  SHOP.TO   753 rows (2023-08-14 -> 2026-08-12)
  AMZN      752 rows (2023-08-14 -> 2026-08-12)
  TSLA      752 rows (2023-08-14 -> 2026-08-12)
  BRK-B     752 rows (2023-08-14 -> 2026-08-12)
  ICE       752 rows (2023-08-14 -> 2026-08-12)
  MRVL      752 rows (2023-08-14 -> 2026-08-12)
  CRM       752 rows (2023-08-14 -> 2026-08-12)
Ingestion complete: 18/18 tickers with price history.


18 signals · 18 tickers with indicators · output → ../output/2026-08-12_105151/


## 2 · Screener (styled)

In [3]:
display_cols = ["Sector","PE","EPS_Growth","Rev_Growth","Debt_Equity","Div_Yield","FCF","Fundamental_Score"]
def style_screen(df):
    if df.empty: return df
    return (df[display_cols].style
        .format({"PE":"{:.1f}","EPS_Growth":"{:.1%}","Rev_Growth":"{:.1%}",
                 "Div_Yield":"{:.2%}","Debt_Equity":"{:.2f}","FCF":"{:,.0f}"}, na_rep="—")
        .background_gradient(subset=["Fundamental_Score"], cmap="Greens", vmin=0, vmax=6)
        .set_caption("Fundamental Screener — sorted by Score (0–6)"))
style_screen(screened_df)

,Sector,PE,EPS_Growth,Rev_Growth,Debt_Equity,Div_Yield,FCF,Fundamental_Score
Ticker,,,,,,,,
GOOGL,Technology,17.2,294.0%,24.2%,0.19,26.00%,"22,665,000,960",6
MU,Technology,20.7,1368.5%,345.7%,0.06,6.00%,"7,639,499,776",6
AMZN,Consumer Discretionary,21.7,242.3%,19.6%,0.46,—,"3,219,124,992",5
META,Technology,22.0,-13.4%,28.0%,0.43,35.00%,"21,553,625,088",5
CRM,Technology,22.4,52.2%,13.3%,1.24,89.00%,"16,552,999,936",5
MSFT,Technology,27.6,31.7%,17.7%,0.29,72.00%,"16,545,500,160",5
TSM,Technology,32.3,77.4%,36.0%,0.15,90.00%,"739,057,991,680",5
NVDA,Technology,34.3,214.5%,85.2%,0.07,46.00%,"46,335,873,024",5
AAPL,Technology,34.6,28.7%,16.4%,0.78,35.00%,"107,721,875,456",5


## 3 · Signal matrix (styled)

**How the scores combine.** Each row fuses two scores into the final action:

- **Fundamental score (0–6):** one point per screener threshold passed.
- **Technical score (0–7, registry-driven):** +1 for each of — price > EMA50, RSI 35–70, a recent bullish MACD crossover, a positive close-regression slope, a **rising EMA50**, volume confirmation, and **price near the lower EMA envelope** (bottom 25% of the band — a mean-reversion entry). The components live in `stockanalysis.signals.TECHNICAL_COMPONENTS`; the max equals its length, so adding/removing a component rescales automatically.
- **Composite** = `0.70·(fund/6) + 0.30·(tech/N)` → **Buy ≥ 0.60 · Hold ≥ 0.40 · Watch** otherwise.
- **Posture** auto-scales with the component count: **Bullish** when score ≥ ⅔·max (≥ 5 of 7), **Bearish** at 0, else **Neutral**.

In [4]:
from stockanalysis.signals import TECHNICAL_COMPONENTS
def style_signals(df):
    if df.empty: return df
    ac = {"Buy":"#1b7837","Hold":"#b8860b","Watch":"#8c8c8c"}
    pc = {"Bullish":"#1b7837","Neutral":"#b8860b","Bearish":"#b2182b"}
    cols = ["Ticker","Sector","Fundamental Score","Technical Posture","Tech Score","Composite","Final Action Signal"]
    return (df[cols].style
        .map(lambda v: f"color:white;font-weight:700;background-color:{ac.get(v,'#8c8c8c')}", subset=["Final Action Signal"])
        .map(lambda v: f"color:{pc.get(v,'#333')};font-weight:600", subset=["Technical Posture"])
        .background_gradient(subset=["Fundamental Score"], cmap="Greens", vmin=0, vmax=6)
        .background_gradient(subset=["Tech Score"], cmap="Greens", vmin=0, vmax=len(TECHNICAL_COMPONENTS))
        .background_gradient(subset=["Composite"], cmap="RdYlGn", vmin=0, vmax=1)
        .format({"Composite":"{:.2f}"}).set_properties(**{"text-align":"center"})
        .set_caption("🎯 Combined Signal Matrix"))
style_signals(signal_matrix)

,Ticker,Sector,Fundamental Score,Technical Posture,Tech Score,Composite,Final Action Signal
0,MU,Technology,6,Neutral,5,0.91,Buy
1,NVDA,Technology,5,Neutral,5,0.80,Buy
2,ASML,Technology,5,Neutral,5,0.80,Buy
3,GOOGL,Technology,6,Neutral,2,0.79,Buy
4,AMZN,Consumer Discretionary,5,Neutral,4,0.76,Buy
5,MSFT,Technology,5,Neutral,4,0.76,Buy
6,TSM,Technology,5,Neutral,4,0.76,Buy
7,AAPL,Technology,5,Neutral,4,0.76,Buy
8,CRM,Technology,5,Neutral,3,0.71,Buy
9,BRK-B,Financials,4,Neutral,5,0.68,Buy


## 4 · Technical dashboards — top 5 picks

The signal matrix is already ranked (**Buy → Hold → Watch**, then Composite descending), so its first rows are the strongest candidates — `signals.top_tickers(signal_matrix, 5)` makes that contract explicit. We build a dashboard for each with `build_technical_dashboard` (returns a Plotly Figure), show them inline **and** write each one as a standalone HTML report into this run's output folder (`../output/<timestamp>/<TICKER>.html`).

Headless equivalent: `sa.run(save_charts=True, top_n=5, out_dir="../output")` — same files, same names, no notebook.

The shaded grey band is the **data-driven EMA envelope** — its lower/upper edges sit at the 2.5th/97.5th percentiles of price's deviation from EMA20, so ~95% of closes fall inside it (the legend shows each ticker's actual ±%, which is asymmetric and volatility-scaled rather than a fixed ±2.5%). Price hugging the lower edge is what feeds the `near_lower_env` technical-score component.

In [ ]:
TOP_N = 5
top = signal_matrix.head(TOP_N)                                   # what the picks look like
tickers = sa.signals.top_tickers(signal_matrix, TOP_N) or list(tech)[:TOP_N]
display(style_signals(top))

chart_paths = []
for t in tickers:
    fig = sa.charts.build_technical_dashboard(t, tech)
    if fig is None:
        print(f"No data for {t}")
        continue
    chart_paths.append(sa.charts.save_html(fig, out / f"{t}.html"))   # standalone HTML report
    fig.show()
print(f"saved {len(chart_paths)} dashboard report(s) → {out}/")

## 5 · Deep fundamental profiles — top 5 picks

Same `tickers` as §4. `build_profile` returns a dict with the raw profile, derived sub-scores, and a ready-to-print `report` string; `profile.save_report` persists that string to `../output/<timestamp>/<TICKER>_profile.txt`, next to the ticker's dashboard.

Headless equivalent: `sa.run(save_profiles=True, top_n=5, out_dir="../output")` (or `stock-analysis run --profiles --top 5`). Each profile costs one extra Yahoo fetch, which is why it's opt-in.

In [ ]:
for t in tickers:                                    # same top 5 as §4
    prof = sa.profile.build_profile(t, screened_df)
    sa.profile.save_report(prof, out / f"{t}_profile.txt")
    print(prof["report"])
print(f"saved {len(tickers)} profile report(s) → {out}/")

## 6 · Daily market overview (Stage 0)

In [7]:
ov = sa.overview.daily_overview(signal_matrix=signal_matrix, tech=tech)
sa.charts.build_index_overview(ov["index_data"]).show()
if ov["vix"]:   print(f"VIX {ov['vix']['value']:.2f} → {ov['vix']['label']}")
pd.DataFrame(ov["indices"]).set_index("Index") if ov["indices"] else None

VIX 14.76 → LOW


,Last,Day %,Week %,YTD %,RSI,Trend
Index,,,,,,
S&P 500,7753.450195,0.326725,0.387133,13.263461,64.854774,Above EMA50
NASDAQ,26618.716797,0.655189,0.968301,14.528560,59.550925,Above EMA50
TSX,36601.699219,0.344887,1.259602,15.416167,68.616418,Above EMA50


## 7 · Export

Persist via the pluggable exporters (Excel works out of the box; Google Sheets needs the `[gsheets]` extra + credentials — see README). This lands next to the dashboard HTML reports from §4, in the same run folder `../output/<timestamp>/`.

In [8]:
sa.outputs.get_exporter("excel", path=str(out / "signal_matrix.xlsx")).export(signal_matrix, screened_df)
print("run folder now holds:", ", ".join(sorted(p.name for p in out.iterdir())))
# sa.outputs.get_exporter("gsheets", spreadsheet="<id|name>").export(signal_matrix, screened_df)

Exported results to '../output/2026-08-12_105151/signal_matrix.xlsx' (sheets: Signal Matrix, Fundamentals).


run folder now holds: AMZN.html, AMZN_profile.txt, ASML.html, ASML_profile.txt, GOOGL.html, GOOGL_profile.txt, MU.html, MU_profile.txt, NVDA.html, NVDA_profile.txt, signal_matrix.xlsx
